# irwkv Hybrid — Colab Linux GPU verification

Select a GPU runtime with compute capability >= 8.0 (for example A100 or L4). A T4 is not suitable for this BF16 path. GPU availability varies. This notebook has not been executed on Colab yet.

Upload the accompanying source ZIP: it includes the local uncommitted implementation, which is not yet on the GitHub branch. No Drive mount or credentials are needed. All project Python commands use /content/irwkv/.venv. Installation preserves repository pins and stops on failure; it does not silently substitute PyTorch versions.

Run cells in order. Complete small-model tests precede optional target-width workloads. This is full-Hybrid validation, not a single-layer benchmark.


In [ ]:
from google.colab import files
from pathlib import Path
import os, subprocess, zipfile
uploaded = files.upload()
archives = [name for name in uploaded if name.endswith(".zip")]
assert len(archives) == 1, "Upload exactly one irwkv source ZIP"
root = Path("/content/irwkv")
assert not root.exists(), "Existing project found; use a fresh runtime to avoid overwriting it"
root.mkdir()
with zipfile.ZipFile(archives[0]) as archive:
    for item in archive.infolist():
        target = (root / item.filename).resolve()
        assert target.is_relative_to(root.resolve()), "Unsafe archive path"
    archive.extractall(root)
os.chdir(root)
logs = root / "validation-logs"
logs.mkdir()
def run(args, name, env=None):
    merged = os.environ.copy()
    merged.update(env or {})
    merged["PATH"] = str(root / ".venv/bin") + os.pathsep + merged["PATH"]
    merged["SETUPTOOLS_SCM_PRETEND_VERSION"] = "0.0.0"
    with (logs / (name + ".log")).open("w") as log:
        process = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, env=merged)
        for line in process.stdout:
            print(line, end="")
            log.write(line)
        status = process.wait()
    if status:
        raise RuntimeError(f"{name} failed ({status}); inspect its log before continuing")
run(["nvidia-smi"], "gpu")
run(["nvcc", "--version"], "cuda-toolkit")



In [ ]:
run(["bash", "-lc", "curl -LsSf https://astral.sh/uv/install.sh | sh"], "install-uv")
import shutil
uv = shutil.which("uv") or str(Path.home() / ".local/bin/uv")
run(["bash", "-lc", "curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal"], "install-rust")
os.environ["PATH"] = str(Path.home() / ".cargo/bin") + os.pathsep + os.environ["PATH"]
run([uv, "venv", "--python", "3.12", ".venv"], "venv")
py = str(root / ".venv/bin/python")
run([uv, "pip", "install", "--python", py, "-r", "requirements/build/cuda.txt",
     "--torch-backend=auto"], "build-dependencies")
run([py, "-c", """import torch
assert torch.cuda.is_available(), 'CUDA unavailable'
print(torch.__version__, torch.version.cuda, torch.cuda.get_device_name())
assert torch.cuda.get_device_capability()[0] >= 8, 'BF16 path requires Ampere or newer'
assert torch.cuda.is_bf16_supported(), 'BF16 unavailable'
"""], "torch-gpu")
run([uv, "pip", "install", "--python", py, "-r", "requirements/rwkv.txt",
     "--torch-backend=auto"], "runtime-dependencies")
run([uv, "pip", "install", "--python", py, "-e", ".", "--no-build-isolation",
     "--torch-backend=auto"], "build-irwkv", {"MAX_JOBS": "2"})
run([uv, "pip", "install", "--python", py, "pytest"], "test-dependencies")
run([uv, "pip", "freeze", "--python", py], "packages")



## Complete-model correctness and actual engine smoke

The explicit Flash test is required here. A failure is a failure, not a skipped success. The engine workload below covers concurrent sessions and multiple turns with prefix reuse; it is a smoke test, not a proof of scheduler/preemption correctness or throughput at target scale.


In [ ]:
run([py, "-m", "pytest", "--noconftest",
     "tests/models/rwkv/test_rwkv7_hybrid.py", "-q", "-rs"],
    "full-hybrid-correctness", {"IRWKV_TEST_FLASH_HYBRID": "1"})
run([py, "-m", "tools.export_rwkv7_hybrid", "exported/hybrid-smoke",
     "--width", "64", "--layers", "4", "--vocab-size", "4096"], "export-smoke")
run([py, "-m", "benchmarks.benchmark_rwkv7_hybrid_agentic", "exported/hybrid-smoke",
     "--concurrency", "2", "--sessions", "2", "--system", "64", "--context", "128",
     "--tool", "32", "--output", "8", "--turns", "4",
     "--max-model-len", "1024", "--max-num-batched-tokens", "128"], "engine-smoke")



## Optional target-width Agent workload

Disabled by default: inspect the available GPU memory and successful smoke results first. BF16 weights alone require approximately 13.3/16.6/20.3 GiB for widths 3328/3584/3840, plus KV cache, recurrent state, activations and workspace. Start with concurrency 1. Default author prompt/turn lengths are preserved. Published B300 FP8 numbers are not comparable to this BF16 eager adapter. Colab hardware and runtime duration are not guaranteed.


In [ ]:
RUN_TARGET_WORKLOAD = False
WIDTH = 3328
CONCURRENCY = "1"
if RUN_TARGET_WORKLOAD:
    assert WIDTH in (3328, 3584, 3840)
    model_path = f"exported/hybrid-{WIDTH}"
    run([py, "-m", "tools.export_rwkv7_hybrid", model_path, "--width", str(WIDTH)],
        f"export-{WIDTH}")
    run([py, "-m", "benchmarks.benchmark_rwkv7_hybrid_agentic", model_path,
         "--concurrency", CONCURRENCY, "--slo", "30",
         "--tag", f"colab-irwkv-bf16-{WIDTH}"], f"agentic-{WIDTH}")



## Download logs (also run after any failed cell)

Logs include hardware, installed versions and exact errors. Review them before sharing. Download before the Colab runtime is recycled.


In [ ]:
import shutil
bundle = shutil.make_archive("/content/irwkv-validation-logs", "zip", logs)
files.download(bundle)

